### Schema inspection & dataset type detection

Column normalization and synonym handling

In [2]:
import pandas as pd
df=pd.read_csv('dataset1_text_rich_transactions.csv')

In [3]:
df.head()

,transaction_id,transaction_date,amount,currency,description,vendor_name,gst_applicable,gst_slab,itc_eligible,category_label,is_anomaly
0,TXN0000001,01-04-2024,7641.92,INR,imps ikea rcpt/2404/001 subs - office supplies...,IKEA,True,12%,TRUE,Office Supplies,0
1,TXN0000002,01-04-2024,58560.14,INR,upi awfis rcpt/2404/002 fy24 - rent exp# ref ...,Awfis,True,18%,TRUE,Rent,0
2,TXN0000003,01-04-2024,2184.95,INR,/ imps zomato inv/2404/003 q2 - meals exp,ZOMATO,True,5%,FALSE,Meals,0
3,TXN0000004,01-04-2024,5375.79,INR,*CARD OFFICE DEPOT Pvt Ltd INV/2404/004 FY25 -...,OFFICE DEPOT Pvt Ltd,True,12%,TRUE,Office Supplies,0
4,TXN0000005,01-04-2024,68830.10,INR,-UPI AWFIS INDIA PVT LTD TAXINV/2404/005 RENT ...,AWFIS INDIA PVT LTD,True,18%,TRUE,Rent,0


In [4]:
import re
from typing import Dict, Optional, Tuple

# Canonical roles your pipelines expect
CANONICAL_ROLES = [
    "transaction_id",
    "description",
    "vendor_name",
    "amount",
    "transaction_date",
    "hsn_code",
    "sac_code",
    "currency",
]

# Synonym patterns for each role (extend as you see more data)
ROLE_SYNONYMS = {
    "transaction_id": [
        r"^txn_?id$", r"^transaction_?id$", r"^id$",r"^doc_?id$",
        r"^voucher_?id$", r"^doc_?no$", r"^document_?number$",
    ],
    "description": [
        r"^description$", r"^narration$", r"^details?$",
        r"^remark$", r"^remarks$", r"^memo$",
    ],
    "vendor_name": [
        r"^vendor$", r"^vendor_?name$", r"^party_?name$",
        r"^supplier$", r"^customer$", r"^counterparty$",
    ],
    "amount": [
        r"^amount$", r"^txn_?amount$", r"^transaction_?amount$",r"^cost_?to_?company$",
        r"^debit$", r"^credit$", r"^value$", r"^net_?amount$",
    ],
    "transaction_date": [
        r"^date$", r"^txn_?date$", r"^transaction_?date$",
        r"^posting_?date$", r"^voucher_?date$", r"^doc_?date$",
    ],
    "hsn_code": [
        r"^hsn$", r"^hsn_?code$", r"^product_?hsn$",
    ],
    "sac_code": [
        r"^sac$", r"^sac_?code$", r"^service_?sac$",
    ],
    "currency": [
        r"^currency$", r"^ccy$", r"^curr_?code$", r"^iso_?currency$",
    ],
}

def normalize_name(col: str) -> str:
    """Simplify column name for matching: lower, strip, remove spaces and punctuation."""
    c = col.strip().lower()
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"[^a-z0-9_]", "", c)
    return c

def match_role(col_norm: str) -> Optional[str]:
    """Return canonical role for a normalized column name, or None if no match."""
    # Direct exact matches
    for role in CANONICAL_ROLES:
        if col_norm == role:
            return role

    # Regex-based synonym matching
    for role, patterns in ROLE_SYNONYMS.items():
        for pat in patterns:
            if re.match(pat, col_norm):
                return role

    return None


Role mapping + dataset type detection

In [5]:
from typing import List

def inspect_schema(columns: List[str]) -> Tuple[str, Dict[str, str]]:
    """
    Inspect incoming column names and:
      - infer dataset type
      - return mapping: raw_col_name -> canonical_role

    Returns:
      dataset_type: one of ["RICH_TEXT", "DESC_ONLY", "NO_DESC", "HSN_SAC", "UNSUPPORTED"]
      role_map: dict mapping raw column names -> canonical role
    """
    role_map: Dict[str, str] = {}
    for col in columns:
        col_norm = normalize_name(col)
        role = match_role(col_norm)
        if role is not None:
            role_map[col] = role

    found_roles = set(role_map.values())

    has_desc = "description" in found_roles
    has_vendor = "vendor_name" in found_roles
    has_amount = "amount" in found_roles
    has_date = "transaction_date" in found_roles
    has_hsn = "hsn_code" in found_roles
    has_sac = "sac_code" in found_roles

    # Type 4: HSN / SAC-based
    if (has_hsn or has_sac) and has_amount and has_date:
        dataset_type = "HSN_SAC"
    # Type 1: Rich text (description + vendor + amount + date)
    elif has_desc and has_vendor and has_amount and has_date:
        dataset_type = "RICH_TEXT"
    # Type 2: Description only (no vendor)
    elif has_desc and (not has_vendor) and has_amount and has_date:
        dataset_type = "DESC_ONLY"
    # Type 3: No description (amount + date, maybe id/vendor_id)
    elif (not has_desc) and has_amount and has_date:
        dataset_type = "NO_DESC"
    else:
        dataset_type = "UNSUPPORTED"

    return dataset_type, role_map





# Usage inside your backend / notebook:
# cols = list(df.columns)
# dataset_type, role_map = inspect_schema(cols)

# print("Detected dataset type:", dataset_type)
# print("Role map:", role_map)



### Regex (for speed) + Fuzzy Matching (for typos)

In [6]:
import re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import pandas as pd

from sentence_transformers import SentenceTransformer, util  # new import

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
_embedding_model = None  # lazy-loaded

try:
    from rapidfuzz import process, fuzz
    FUZZY_AVAILABLE = True
except ImportError:
    FUZZY_AVAILABLE = False


# Canonical roles your ML pipelines expect
CANONICAL_ROLES = [
    "transaction_id",
    "description",
    "vendor_name",
    "amount",
    "transaction_date",
    "hsn_code",
    "sac_code",
    "currency",
    "gst_applicable",
    "gst_slab",
    "itc_eligible",
    "category_label",
]

# Regex-based synonyms
ROLE_SYNONYMS = {
    "transaction_id": [
        r"^txn_?id$", r"^transaction_?id$", r"^voucher_?id$", r"^doc_?no$", r"^doc_?id$",
        r"^document_?number$", r"^ref_?no$", r"^id$",
    ],
    "description": [
        r"^description$", r"^narration$", r"^details?$",
        r"^remark$", r"^remarks$", r"^memo$", r"^particulars$",
    ],
    "vendor_name": [
        r"^vendor$", r"^vendor_?name$", r"^party_?name$",
        r"^supplier$", r"^customer$", r"^counterparty$",
        r"^payee$", r"^account_?name$",
    ],
    "amount": [
        r"^amount$", r"^txn_?amount$", r"^txn_?amt$",r"^transaction_?amount$", r"^cost_?to_?company$",
        r"^debit$", r"^credit$", r"^value$", r"^net_?amount$", r"^total$",
    ],
    "transaction_date": [
        r"^date$", r"^txn_?date$", r"^transaction_?date$",
        r"^posting_?date$", r"^voucher_?date$", r"^doc_?date$", r"^entry_?date$",
    ],
    "hsn_code": [
        r"^hsn$", r"^hsn_?code$", r"^product_?hsn$", r"^hsn_?sac$",
    ],
    "sac_code": [
        r"^sac$", r"^sac_?code$", r"^service_?sac$", r"^hsn_?sac$",
    ],
    "currency": [
        r"^currency$", r"^ccy$", r"^curr_?code$", r"^iso_?currency$", r"^curr$",
    ],
    "gst_applicable": [
        r"^gst_?applicable$", r"^is_?gst$", r"^gst_?flag$", r"^has_?gst$",
    ],
    "gst_slab": [
        r"^gst_?slab$", r"^gst_?rate$", r"^gst_?percent$", r"^tax_?rate$",
    ],
    "itc_eligible": [
        r"^itc_?eligible$", r"^is_?itc$", r"^itc_?flag$", r"^input_?tax_?credit$",
    ],
    "category_label": [
        r"^category_?label$", r"^category$", r"^classification$",
        r"^expense_?category$", r"^account_?type$",
    ],
}


def normalize_name(col: str) -> str:
    c = col.strip().lower()
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"[^a-z0-9_]", "", c)
    return c.strip("_")


def match_role_exact(col_norm: str) -> Optional[str]:
    for role in CANONICAL_ROLES:
        if col_norm == role:
            return role
    return None


def match_role_regex(col_norm: str) -> Optional[str]:
    for role, patterns in ROLE_SYNONYMS.items():
        for pat in patterns:
            if re.match(pat, col_norm):
                return role
    return None


def match_role_fuzzy(col_norm: str, threshold: int = 85) -> Optional[Tuple[str, int]]:
    if not FUZZY_AVAILABLE:
        return None
    best_match, score, _ = process.extractOne(
        col_norm, CANONICAL_ROLES, scorer=fuzz.token_sort_ratio
    )
    if score >= threshold:
        return best_match, score
    return None


def get_embedding_model() -> SentenceTransformer:
    """
    Lazy-load the sentence transformer model (loaded once per process).
    """
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    return _embedding_model


def match_role_embedding(
    col_norm: str,
    threshold: float = 0.55,
) -> Optional[Tuple[str, float]]:
    """
    Step 4 – semantic matching:
    Use sentence-transformers to map semantically related names
    (e.g., 'cost_to_company' → 'amount', 'party' → 'vendor_name').

    Returns (best_role, similarity) if similarity >= threshold, else None.
    """
    model = get_embedding_model()

    # Encode incoming column name
    q_emb = model.encode(col_norm, convert_to_tensor=True, normalize_embeddings=True)

    # Encode canonical roles once per call (small list, OK)
    role_embeddings = model.encode(
        CANONICAL_ROLES,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    # Compute cosine similarities
    sims = util.cos_sim(q_emb, role_embeddings)[0]  # shape: [num_roles]

    # Find best index
    best_idx = int(sims.argmax().item())
    best_sim = float(sims[best_idx].item())
    best_role = CANONICAL_ROLES[best_idx]
    
    # DEBUG
    print(f"[EMB-DEBUG] '{col_norm}' best={best_role}, sim={best_sim:.3f}")

    if best_sim >= threshold:
        return best_role, best_sim
    return None



@dataclass
class ColumnMapping:
    raw_name: str
    matched_role: Optional[str]
    normalized_name: str
    match_method: str  # "exact" | "regex" | "fuzzy" | "unmapped"
    confidence_score: float


@dataclass
class SchemaInspectionResult:
    dataset_type: str            # "RICH_TEXT" | "DESC_ONLY" | "NO_DESC" | "HSN_SAC" | "UNSUPPORTED"
    column_mappings: List[ColumnMapping]
    role_map: Dict[str, str]     # raw -> canonical
    unmapped_columns: List[str]
    confidence: float            # 0-1 overall


C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:

def inspect_schema_hybrid(
    columns: List[str],
    use_fuzzy: bool = True,
    use_embeddings: bool = False,   # NEW FLAG
    verbose: bool = False,
) -> SchemaInspectionResult:

    role_map: Dict[str, str] = {}
    mappings: List[ColumnMapping] = []
    unmapped: List[str] = []

    for col in columns:
        col_norm = normalize_name(col)

        # 1) Exact
        role = match_role_exact(col_norm)
        if role:
            role_map[col] = role
            mappings.append(ColumnMapping(col, role, col_norm, "exact", 1.0))
            if verbose:
                print(f"[EXACT] '{col}' -> {role}")
            continue

        # 2) Regex
        role = match_role_regex(col_norm)
        if role:
            role_map[col] = role
            mappings.append(ColumnMapping(col, role, col_norm, "regex", 0.9))
            if verbose:
                print(f"[REGEX] '{col}' -> {role}")
            continue

        # 3) Fuzzy
        if use_fuzzy:
            res = match_role_fuzzy(col_norm)
            if res:
                role, score = res
                conf = score / 100.0
                role_map[col] = role
                mappings.append(ColumnMapping(col, role, col_norm, "fuzzy", conf))
                if verbose:
                    print(f"[FUZZY] '{col}' -> {role} (score={score})")
                continue

        # 4) Embeddings (semantic) – ONLY if exact, regex, fuzzy all failed
        if use_embeddings:
            res = match_role_embedding(col_norm, threshold=0.50)
            if res:
                role, sim = res
                conf = float(sim)  # sim already in [0,1] because of normalized embeddings
                role_map[col] = role
                mappings.append(ColumnMapping(col, role, col_norm, "embedding", conf))
                if verbose:
                    print(f"[EMBEDDING] '{col}' -> {role} (sim={sim:.2f})")
                continue

        # 5) Unmapped
        unmapped.append(col)
        mappings.append(ColumnMapping(col, None, col_norm, "unmapped", 0.0))
        if verbose:
            print(f"[UNMAPPED] '{col}'")


    # Detect dataset type
    found = set(role_map.values())
    has_desc = "description" in found
    has_vendor = "vendor_name" in found
    has_amount = "amount" in found
    has_date = "transaction_date" in found
    has_hsn = "hsn_code" in found or "sac_code" in found

    if has_hsn and has_amount and has_date:
        dataset_type = "HSN_SAC"
    elif has_desc and has_vendor and has_amount and has_date:
        dataset_type = "RICH_TEXT"
    elif has_desc and (not has_vendor) and has_amount and has_date:
        dataset_type = "DESC_ONLY"
    elif (not has_desc) and has_amount and has_date:
        dataset_type = "NO_DESC"
    else:
        dataset_type = "UNSUPPORTED"

    # Overall confidence = (mapped fraction) * avg(conf)
    if not mappings:
        overall_conf = 0.0
    else:
        mapped = [m for m in mappings if m.matched_role]
        if mapped:
            avg_conf = sum(m.confidence_score for m in mapped) / len(mapped)
            overall_conf = (len(mapped) / len(mappings)) * avg_conf
        else:
            overall_conf = 0.0

    return SchemaInspectionResult(
        dataset_type=dataset_type,
        column_mappings=mappings,
        role_map=role_map,
        unmapped_columns=unmapped,
        confidence=overall_conf,
    )


### Helpers to use role map

In [8]:
def get_col(role_map: Dict[str, str], role: str) -> Optional[str]:
    for raw, r in role_map.items():
        if r == role:
            return raw
    return None


def build_model_input(df_raw: pd.DataFrame, role_map: Dict[str, str]) -> pd.DataFrame:
    """
    Build a DataFrame whose columns are the canonical roles
    (transaction_id, description, vendor_name, ...) that your model expects.
    Any role that is not present in role_map will simply be missing.
    """
    df = pd.DataFrame(index=df_raw.index)
    for raw_col, role in role_map.items():
        # role is already one of CANONICAL_ROLES
        df[role] = df_raw[raw_col]
    return df


### Orchestration: choose pipeline by dataset type

In [9]:
from dataclasses import asdict

@dataclass
class PredictionRecord:
    transaction_id: str
    predicted_category: Optional[str]
    category_confidence: float
    gst_rate_pred: Optional[float]
    itc_eligible_pred: Optional[str]
    anomaly_flag: Optional[int]   # -1 / 1 or None
    anomaly_score: Optional[float]


@dataclass
class OrchestratorOutput:
    dataset_type: str
    schema_confidence: float
    unmapped_columns: List[str]
    predictions: List[PredictionRecord]
    messages: List[str]


# -------------------------------------------------------------------
# Pipeline implementations (replace stubs with real models)
# -------------------------------------------------------------------
def run_rich_text_pipeline(df: pd.DataFrame) -> List[PredictionRecord]:
    """
    Type 1 (RICH_TEXT): df columns are canonical:
    transaction_id, description, vendor_name, amount, transaction_date, ...
    TODO: replace stub with your TF-IDF + XGBoost + anomaly detection.
    """
    tid_col = "transaction_id" if "transaction_id" in df.columns else df.columns[0]
    preds: List[PredictionRecord] = []
    for i, row in df.head(5).iterrows():
        preds.append(
            PredictionRecord(
                transaction_id=str(row.get(tid_col, f"TXN{i}")),
                predicted_category="IT Services",
                category_confidence=0.9,
                gst_rate_pred=18.0,
                itc_eligible_pred="Yes",
                anomaly_flag=1,
                anomaly_score=0.05,
            )
        )
    return preds


def run_desc_only_pipeline(df: pd.DataFrame) -> List[PredictionRecord]:
    """
    Type 2 (DESC_ONLY): description + amount + transaction_date.
    """
    tid_col = "transaction_id" if "transaction_id" in df.columns else df.columns[0]
    preds: List[PredictionRecord] = []
    for i, row in df.head(5).iterrows():
        preds.append(
            PredictionRecord(
                transaction_id=str(row.get(tid_col, f"TXN{i}")),
                predicted_category="Office Supplies",
                category_confidence=0.65,
                gst_rate_pred=12.0,
                itc_eligible_pred="Yes",
                anomaly_flag=1,
                anomaly_score=0.15,
            )
        )
    return preds


def run_no_desc_pipeline(df: pd.DataFrame) -> List[PredictionRecord]:
    """
    Type 3 (NO_DESC): amount + transaction_date only.
    """
    tid_col = "transaction_id" if "transaction_id" in df.columns else df.columns[0]
    preds: List[PredictionRecord] = []
    for i, row in df.head(5).iterrows():
        preds.append(
            PredictionRecord(
                transaction_id=str(row.get(tid_col, f"TXN{i}")),
                predicted_category="Rent",
                category_confidence=0.45,
                gst_rate_pred=18.0,
                itc_eligible_pred="Unknown",
                anomaly_flag=1,
                anomaly_score=0.2,
            )
        )
    return preds


def run_hsn_sac_pipeline(df: pd.DataFrame) -> List[PredictionRecord]:
    """
    Type 4 (HSN_SAC): HSN/SAC rule-based.
    """
    tid_col = "transaction_id" if "transaction_id" in df.columns else df.columns[0]
    preds: List[PredictionRecord] = []
    for i, row in df.head(5).iterrows():
        preds.append(
            PredictionRecord(
                transaction_id=str(row.get(tid_col, f"TXN{i}")),
                predicted_category="Software",
                category_confidence=1.0,
                gst_rate_pred=18.0,
                itc_eligible_pred="Yes",
                anomaly_flag=1,
                anomaly_score=0.1,
            )
        )
    return preds

### Main orchestration entrypoint

In [10]:
def run_core_pipeline(df_raw: pd.DataFrame, verbose: bool = False) -> OrchestratorOutput:
    """
    Full backend pipeline:
      - schema inspection
      - dataset type detection
      - canonicalize to model schema
      - route to appropriate model
    """
    cols = df_raw.columns.tolist()
    inspect_result = inspect_schema_hybrid(cols, use_fuzzy=True, use_embeddings=True, verbose=verbose)

    if verbose:
        print(f"\n[SCHEMA] Detected type: {inspect_result.dataset_type}")
        print(f"[SCHEMA] Confidence: {inspect_result.confidence:.1%}")
        print(f"[SCHEMA] Unmapped: {inspect_result.unmapped_columns}")

    ds_type = inspect_result.dataset_type
    role_map = inspect_result.role_map
    preds: List[PredictionRecord] = []
    messages: List[str] = []

    # Build canonical-view DataFrame with columns equal to canonical roles
    canon_df = build_model_input(df_raw, role_map)

    if ds_type == "RICH_TEXT":
        messages.append("Processed as Type 1: Rich Text (description + vendor + amount + date).")
        preds = run_rich_text_pipeline(canon_df)

    elif ds_type == "DESC_ONLY":
        messages.append("Processed as Type 2: Description Only (no vendor).")
        preds = run_desc_only_pipeline(canon_df)

    elif ds_type == "NO_DESC":
        messages.append("Processed as Type 3: No Description (numeric/time only).")
        preds = run_no_desc_pipeline(canon_df)

    elif ds_type == "HSN_SAC":
        messages.append("Processed as Type 4: HSN/SAC-based deterministic mapping.")
        preds = run_hsn_sac_pipeline(canon_df)

    else:
        messages.append("Unsupported schema: cannot map to any known dataset type.")

    return OrchestratorOutput(
        dataset_type=ds_type,
        schema_confidence=inspect_result.confidence,
        unmapped_columns=inspect_result.unmapped_columns,
        predictions=preds,
        messages=messages,
    )

### Example API-style response

In [11]:
def serialize_output(output: OrchestratorOutput) -> dict:
    return {
        "dataset_type": output.dataset_type,
        "schema_confidence": output.schema_confidence,
        "unmapped_columns": output.unmapped_columns,
        "messages": output.messages,
        "predictions": [asdict(p) for p in output.predictions],
    }


# Example usage:
if __name__ == "__main__":
    df_demo = pd.DataFrame({
        "doc_ID": ["TXN1", "TXN2"],
        "TxnDate": ["2024-01-01", "2024-01-02"],
        "Narration": ["rent payment", "office supplies"],
        "vender_name": ["WeWork", "Office Depot"],
        "cost_to_company": [50000, 3000],
    })

    out = run_core_pipeline(df_demo, verbose=True)
    print(serialize_output(out))


[REGEX] 'doc_ID' -> transaction_id
[REGEX] 'TxnDate' -> transaction_date
[REGEX] 'Narration' -> description
[FUZZY] 'vender_name' -> vendor_name (score=90.9090909090909)
[REGEX] 'cost_to_company' -> amount

[SCHEMA] Detected type: RICH_TEXT
[SCHEMA] Confidence: 90.2%
[SCHEMA] Unmapped: []
{'dataset_type': 'RICH_TEXT', 'schema_confidence': 0.9018181818181817, 'unmapped_columns': [], 'messages': ['Processed as Type 1: Rich Text (description + vendor + amount + date).'], 'predictions': [{'transaction_id': 'TXN1', 'predicted_category': 'IT Services', 'category_confidence': 0.9, 'gst_rate_pred': 18.0, 'itc_eligible_pred': 'Yes', 'anomaly_flag': 1, 'anomaly_score': 0.05}, {'transaction_id': 'TXN2', 'predicted_category': 'IT Services', 'category_confidence': 0.9, 'gst_rate_pred': 18.0, 'itc_eligible_pred': 'Yes', 'anomaly_flag': 1, 'anomaly_score': 0.05}]}


In [12]:
out = run_core_pipeline(df, verbose=True)
print(serialize_output(out))

[EXACT] 'transaction_id' -> transaction_id
[EXACT] 'transaction_date' -> transaction_date
[EXACT] 'amount' -> amount
[EXACT] 'currency' -> currency
[EXACT] 'description' -> description
[EXACT] 'vendor_name' -> vendor_name
[EXACT] 'gst_applicable' -> gst_applicable
[EXACT] 'gst_slab' -> gst_slab
[EXACT] 'itc_eligible' -> itc_eligible
[EXACT] 'category_label' -> category_label
[EMB-DEBUG] 'is_anomaly' best=transaction_id, sim=0.268
[UNMAPPED] 'is_anomaly'

[SCHEMA] Detected type: RICH_TEXT
[SCHEMA] Confidence: 90.9%
[SCHEMA] Unmapped: ['is_anomaly']
{'dataset_type': 'RICH_TEXT', 'schema_confidence': 0.9090909090909091, 'unmapped_columns': ['is_anomaly'], 'messages': ['Processed as Type 1: Rich Text (description + vendor + amount + date).'], 'predictions': [{'transaction_id': 'TXN0000001', 'predicted_category': 'IT Services', 'category_confidence': 0.9, 'gst_rate_pred': 18.0, 'itc_eligible_pred': 'Yes', 'anomaly_flag': 1, 'anomaly_score': 0.05}, {'transaction_id': 'TXN0000002', 'predicted